# 02 · Reproduce the external test results

Loads the fine-tuned model from `models/amp_bert/` and evaluates it on the **external**
test set = `veltri_dramp_cdhit_90.csv` (AMP) ∪ `non_amp_ampep_cdhit90.csv` (non-AMP).

Run `01_train_amp_bert.ipynb` first (or drop a pretrained checkpoint into `models/amp_bert/`).

In [ ]:
# --- bootstrap: works locally and on Google Colab ---
# Pin a transformers version with the classic BERT tokenizer. The transformers
# shipped on Colab is too new and mishandles ProtBERT-BFD's WordPiece tokenizer.
import sys, subprocess, pathlib
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.46.3', 'tokenizers>=0.20,<0.21', 'sentencepiece'],
               check=True)

REPO = "https://github.com/BioGavin/AMP-BERT.git"
NAME = "AMP-BERT"
try:
    import amp_bert  # already installed / on sys.path
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((c for c in (here, here.parent, here / NAME)
                 if (c / 'src' / 'amp_bert').exists()), None)
    if root is None:
        target = here / NAME
        if target.exists():            # stale/partial clone -> refresh
            subprocess.run(['git', '-C', str(target), 'pull'], check=True)
        else:
            subprocess.run(['git', 'clone', '--depth', '1', REPO], check=True)
        root = target
    sys.path.insert(0, str(root / 'src'))

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
import pandas as pd
from transformers import Trainer, AutoModelForSequenceClassification
from amp_bert import AmpDataset, load_dataset, compute_metrics, build_training_args
from amp_bert.config import TEST_AMP_CSV, TEST_NONAMP_CSV, MODELS_DIR, RESULTS_DIR

## Build the merged external test set

In [ ]:
amp_df = load_dataset(TEST_AMP_CSV)
nonamp_df = load_dataset(TEST_NONAMP_CSV)
test_df = pd.concat([amp_df, nonamp_df])
print('AMP:', len(amp_df), '| non-AMP:', len(nonamp_df), '| total:', len(test_df))
test_df['AMP'].value_counts()

In [ ]:
test_dataset = AmpDataset(test_df, max_len=200)

## Load fine-tuned model & predict

In [ ]:
model_path = MODELS_DIR / 'amp_bert'
assert model_path.exists(), f'No model at {model_path}. Run 01_train_amp_bert.ipynb first.'
model = AutoModelForSequenceClassification.from_pretrained(str(model_path))

eval_args = build_training_args(str(RESULTS_DIR / 'amp_bert_test'), do_train=False,
                                per_device_eval_batch_size=56)
trainer = Trainer(model=model, args=eval_args, compute_metrics=compute_metrics)

In [ ]:
predictions, label_ids, metrics = trainer.predict(test_dataset)
metrics

## Save predictions for inspection

In [ ]:
import numpy as np
test_df = test_df.copy()
test_df['pred'] = predictions.argmax(-1)
out_csv = RESULTS_DIR / 'external_test_predictions.csv'
test_df.to_csv(out_csv)
print('wrote', out_csv)